In [2]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
from arch import arch_model
from typing import Dict
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
try:
    # Works in a standard .py script
    root = Path(__file__).resolve().parent.parent
except NameError:
    # Works in a Jupyter notebook (.ipynb)
    root = Path.cwd().parent 

# Add project root to sys.path to enable 'from src.data import ...'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Now your imports will work
from src.data import load_dataframe, save_dataframe
from src.models.garch import (
    estimate_garch_multivariate,
    extract_volatilities,
    extract_standardized_residuals,
    garch_diagnostics,
    get_garch_parameters
)
from src.models.dcc import estimate_dcc, dcc_diagnostics, estimate_ccc
from src.models.lrmes import (
    simulate_garch_dcc_returns,
    calculate_lrmes,
    lrmes_diagnostics
)
from src.utils.config import CONFIG, PROCESSED_DATA_DIR, RESULTS_DATA_DIR
import pandas as pd


def load_processed_data():
    """Load processed returns and bank universe."""
    print("\n" + "=" * 60)
    print("LOADING PROCESSED DATA")
    print("=" * 60)

    returns = load_dataframe("returns_clean", PROCESSED_DATA_DIR)
    universe = load_dataframe("banks_universe_clean", PROCESSED_DATA_DIR)

    print(f"\nReturns shape: {returns.shape}")
    print(f"Date range: {returns.index.min()} to {returns.index.max()}")
    print(f"Banks: {len(universe)}")

    return returns, universe


def estimate_garch_models(returns_df):
    """Estimate GJR-GARCH(1,1) for all instruments."""
    print("\n" + "=" * 60)
    print("ESTIMATING GJR-GARCH(1,1) MODELS")
    print("=" * 60)
    print(f"Estimating for {len(returns_df.columns)} instruments...")
    print()

    garch_results = estimate_garch_multivariate(
        returns_df,
        p=CONFIG.GARCH_P,
        q=CONFIG.GARCH_Q,
        mean_model="Constant",
        dist="normal",
        verbose=True
    )

    # Extract diagnostics
    diagnostics = garch_diagnostics(garch_results)

    print(f"\n{diagnostics['convergence'].sum()}/{len(diagnostics)} models converged")
    print(f"Average persistence: {diagnostics['persistence'].mean():.4f}")
    avg_p = diagnostics['ljung_box_p'].apply(pd.Series).mean()
    print(f"Average Ljung-Box p-values:\n{avg_p.round(4)}")

    # Save diagnostics
    metadata = {
        "description": "GARCH estimation diagnostics",
        "model": "GJR-GARCH(1,1)",
        "n_instruments": len(garch_results),
        "convergence_rate": float(diagnostics["convergence"].sum() / len(diagnostics))
    }
    diagnostics.drop('ljung_box_p', axis=1, inplace=True) # remove p-values from diagnostics as they are vectors
    save_dataframe(diagnostics, "garch_diagnostics", RESULTS_DATA_DIR, metadata)

    # Save parameters
    params = get_garch_parameters(garch_results)
    save_dataframe(params, "garch_parameters", RESULTS_DATA_DIR, {"description": "GARCH parameters"})

    # Save conditional volatilities
    vols = extract_volatilities(garch_results)
    save_dataframe(vols, "conditional_volatilities", RESULTS_DATA_DIR, {"description": "GARCH conditional volatilities"})

    return garch_results


def estimate_dcc_model(garch_results):
    """Estimate Dynamic Conditional Correlation."""
    print("\n" + "=" * 60)
    print("ESTIMATING DYNAMIC CONDITIONAL CORRELATION (DCC)")
    print("=" * 60)

    # Extract standardized residuals
    std_resids = extract_standardized_residuals(garch_results)

    print(f"Standardized residuals shape: {std_resids.shape}")
    print(f"Mean: {std_resids.mean().mean():.4f} (should be ~0)")
    print(f"Std: {std_resids.std().mean():.4f} (should be ~1)")

    # Estimate DCC
    try:
        print("\nEstimating DCC...")
        dcc_result = estimate_dcc(std_resids, initial_params=[0.01, 0.95])

        if dcc_result["success"]:
            a, b = dcc_result["params"]
            print(f"✓ DCC converged: a={a:.4f}, b={b:.4f}, persistence={a+b:.4f}")
        else:
            print(f"⚠ DCC did not fully converge: {dcc_result['message']}")

        # Diagnostics
        dcc_diag = dcc_diagnostics(dcc_result, list(std_resids.columns))
        print(f"Average correlation: {dcc_diag['avg_correlation'].iloc[0]:.4f}")
        print(f"Correlation volatility: {dcc_diag['std_correlation'].iloc[0]:.4f}")

        save_dataframe(dcc_diag, "dcc_diagnostics", RESULTS_DATA_DIR, {"description": "DCC diagnostics"})

        return dcc_result

    except Exception as e:
        print(f"✗ DCC estimation failed: {e}")
        print("Falling back to Constant Conditional Correlation (CCC)...")

        ccc_result = estimate_ccc(std_resids)
        print(f"✓ Using CCC (constant correlation)")

        return ccc_result


def calculate_lrmes_estimates(garch_results, dcc_result, instrument_names):
    """Calculate LRMES via Monte Carlo simulation."""
    print("\n" + "=" * 60)
    print("CALCULATING LRMES VIA MONTE CARLO SIMULATION")
    print("=" * 60)
    print(f"Horizon: {CONFIG.CRISIS_HORIZON_WEEKS} weeks (6 months)")
    print(f"Crisis threshold: {CONFIG.CRISIS_THRESHOLD * 100:.0f}% market decline")
    print(f"Number of simulations: {CONFIG.N_SIMULATIONS:,}")
    print()

    # Prepare GARCH parameters
    garch_params = {name: garch_results[name] for name in instrument_names if name in garch_results}

    # Simulate
    print("Running Monte Carlo simulation...")
    simulations = simulate_garch_dcc_returns(
        garch_params=garch_params,
        dcc_result=dcc_result,
        instrument_names=instrument_names,
        horizon=CONFIG.CRISIS_HORIZON_WEEKS,
        n_simulations=CONFIG.N_SIMULATIONS,
        random_seed=CONFIG.RANDOM_SEED
    )

    print("✓ Simulation complete")
    print(f"Simulated returns shape: {simulations['simulated_returns'].shape}")

    # Calculate LRMES
    print("\nCalculating LRMES...")
    lrmes_values = calculate_lrmes(
        simulations=simulations,
        crisis_threshold=CONFIG.CRISIS_THRESHOLD,
        market_idx=0,  # First column is market benchmark
        min_crisis_scenarios=500
    )

    # Diagnostics
    lrmes_diag = lrmes_diagnostics(lrmes_values, simulations)
    print(f"\n✓ LRMES calculated")
    print(f"Crisis scenarios: {lrmes_values['_n_crisis_scenarios']} ({lrmes_values['_crisis_probability']*100:.2f}%)")
    print(f"Average LRMES: {lrmes_diag['avg_lrmes'].iloc[0]:.4f}")
    print(f"Range: [{lrmes_diag['min_lrmes'].iloc[0]:.4f}, {lrmes_diag['max_lrmes'].iloc[0]:.4f}]")

    # Save diagnostics
    save_dataframe(lrmes_diag, "lrmes_diagnostics", RESULTS_DATA_DIR, {"description": "LRMES diagnostics"})

    # Convert to DataFrame
    lrmes_df = pd.DataFrame({
        "instrument": [k for k in lrmes_values.keys() if not k.startswith("_")],
        "lrmes": [lrmes_values[k] for k in lrmes_values.keys() if not k.startswith("_")]
    })

    save_dataframe(lrmes_df, "lrmes_estimates", RESULTS_DATA_DIR, {
        "description": "LRMES estimates",
        "horizon_weeks": CONFIG.CRISIS_HORIZON_WEEKS,
        "crisis_threshold": CONFIG.CRISIS_THRESHOLD,
        "n_simulations": CONFIG.N_SIMULATIONS,
        "n_crisis_scenarios": lrmes_values["_n_crisis_scenarios"]
    })

    return lrmes_df

In [ ]:
returns, universe = load_processed_data()
garch_results = estimate_garch_models(returns)
dcc_result = estimate_dcc_model(garch_results)


LOADING PROCESSED DATA
✓ Loaded returns_clean.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/processed
  Shape: (3772, 40)
✓ Loaded banks_universe_clean.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/processed
  Shape: (40, 3)

Returns shape: (3772, 40)
Date range: 2009-12-29 00:00:00 to 2026-03-09 00:00:00
Banks: 40

ESTIMATING GJR-GARCH(1,1) MODELS
Estimating for 40 instruments...

Estimating GARCH for AJBNK.DU... ✓ AIC: 14098.47
Estimating GARCH for DISB.DU... ✓ AIC: 12185.72
Estimating GARCH for ENBD.DU... ✓ AIC: 13859.39
Estimating GARCH for ADIB.AD... ✓ AIC: 13386.86
Estimating GARCH for FAB.AD... ✓ AIC: 13765.76
Estimating GARCH for BOS.AD... ✓ AIC: 13324.43
Estimating GARCH for SIB.AD... ✓ AIC: 13950.99
Estimating GARCH for ADCB.AD... ✓ AIC: 14248.58
Estimating GARCH for BBKB.BH... ✓ AIC: 8011.36
Estimating GARCH for SALAM.BH... ✓ AIC: 13633.42
Estimating GARCH for NATB.BH... ✓ AIC: 8566.80
Estimating GARCH for DOBK.QA... ✓ AIC: 13140.94
Estimat

/Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/.venv/lib/python3.11/site-packages/numpy/linalg/_linalg.py:2349:RuntimeWarning: divide by zero encountered in slogdet
/Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/.venv/lib/python3.11/site-packages/numpy/linalg/_linalg.py:2349:RuntimeWarning: overflow encountered in slogdet
/Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/.venv/lib/python3.11/site-packages/numpy/linalg/_linalg.py:2349:RuntimeWarning: invalid value encountered in slogdet


In [ ]:
instrument_names = list(returns.columns)